# Quick start

This notebook walks through the core xhycom workflow: opening HYCOM `.ab` files
directly into labelled `xr.Dataset` objects, slicing, and plotting.

Set the paths below to point at your data before running.

In [ ]:
import xhycom
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

DATA_PATH = "/path/to/data/"    # directory with archv.*.ab or archm.*.ab
GRID_PATH = "/path/to/topo/regional.grid"

## Open a single archive snapshot

`open_dataset` auto-detects the file type (archive, grid, bathymetry) from the
`.b` header.  When `grid=` is provided, appropriate `lon`/`lat` coordinates are
attached to each variable based on its staggering point on the Arakawa C-grid.

In [ ]:
ds = xhycom.open_dataset(DATA_PATH + "archv.2020_001_00", grid=GRID_PATH)
ds

Every variable carries CF-style metadata (`long_name`, `units`) read from xhycom's
built-in lookup table:

```python
ds["temp"].attrs   # → {'long_name': 'sea water potential temperature', 'units': 'degC'}
```

Dimension and coordinate attrs are also set:
- `k` / `ki` — layer centres / interfaces, `axis: Z`
- `lon` / `lat` — T-point; `lon_u` / `lat_u` — U-point; `lon_v` / `lat_v` — V-point
- `dens` — target sigma-2 density for each layer

## Slicing

xarray's `isel` (index-based) and `sel` (label-based) selectors work directly.

In [ ]:
# Surface temperature at the first time step
sst = ds["temp"].isel(time=0, k=0)
sst

In [ ]:
# Select by layer density instead of layer index
layer = ds["temp"].isel(time=0).sel(dens=28.0, method="nearest")
print(f"Selected layer k={int(layer.k)}, dens={float(layer.dens):.3f} kg m-3")

In [ ]:
# Spatial slice over a lon/lat bounding box
# (use .where() because lon/lat are 2-D curvilinear coordinates)
mask = (ds.lon > -30) & (ds.lon < 30) & (ds.lat > 50) & (ds.lat < 80)
region = ds["temp"].isel(time=0, k=0).where(mask)

## Plotting

Because `lon` and `lat` are 2-D curvilinear arrays, use `pcolormesh` directly
rather than xarray's `.plot()`.  U-point and V-point variables carry
`lon_u`/`lat_u` and `lon_v`/`lat_v` coordinates.

In [ ]:
sst = ds["temp"].isel(time=0, k=0)

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.pcolormesh(sst.lon.values, sst.lat.values, sst.values,
              cmap="RdYlBu_r", transform=ccrs.PlateCarree())
ax.coastlines()
plt.show()

In [ ]:
# U-point variable — use lon_u / lat_u
u = ds["u-vel."].isel(time=0, k=0)

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.pcolormesh(u.lon_u.values, u.lat_u.values, u.values,
              transform=ccrs.PlateCarree())
ax.coastlines()
plt.show()

## Open the grid and bathymetry

In [ ]:
grid = xhycom.open_dataset(GRID_PATH)
grid

In [ ]:
bathy = xhycom.open_dataset("/path/to/topo/depth_TP2a0.10_04", grid=GRID_PATH)

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.pcolormesh(bathy["depth"].lon.values, bathy["depth"].lat.values,
              bathy["depth"].values, cmap="Blues_r", transform=ccrs.PlateCarree())
ax.coastlines()
plt.show()

## Open a time series

Pass a directory or glob pattern — xhycom discovers all
`archv.` / `archm.YYYY_DDD_HH.[ab]` pairs and concatenates them along `time`.

In [ ]:
ds = xhycom.open_mfdataset(DATA_PATH, grid=GRID_PATH)
ds

In [ ]:
# Time-mean surface salinity
smean = ds["saln"].isel(k=0).mean("time")

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.pcolormesh(smean.lon.values, smean.lat.values, smean.values,
              transform=ccrs.PlateCarree())
ax.coastlines()
plt.show()

## Lazy loading with Dask

By default xhycom reads all data eagerly into memory.  For large archives,
pass `chunks` to enable lazy loading — the `.a` binary files are **not read**
until an explicit `.compute()` (or equivalent) is triggered.

```python
ds = xhycom.open_mfdataset(DATA_PATH, grid=GRID_PATH, chunks={"time": 1})
```

With `chunks={"time": 1}` each monthly file is one Dask chunk.  Only the
`.b` text headers are parsed at open time; the actual arrays remain on disk
until you access them:

```python
# Only reads one month from disk:
ds["temp"].isel(time=0, k=0).compute()

# Reads all months, but in parallel if a Dask scheduler is running:
ds["saln"].isel(k=0).mean("time").compute()
```

Other useful chunk strategies:

| chunks | When to use |
|--------|-------------|
| `{"time": 1}` | Large time series, process one snapshot at a time |
| `{"k": 1}` | Depth-by-depth analysis of a single file |
| `{"time": 1, "k": 1}` | Both simultaneously |

In [ ]:
ds_lazy = xhycom.open_mfdataset(DATA_PATH, grid=GRID_PATH, chunks={"time": 1})
ds_lazy   # DataArrays show 'dask.array' — nothing read yet

In [ ]:
# Trigger computation for one time step only
ds_lazy["temp"].isel(time=0, k=0).compute()